# Perbandingan CatBoost dan Logistic Regression URL/QR v9
Mengikuti [protokol sebelum training](ai-agent-docs/model-family-comparison-v9-plan.md). Dua konfigurasi per keluarga dilatih pada 535.504 URL yang sama. Representasi bersama: 64 komponen SVD teks + 38 fitur struktur. Frozen LR lengkap v8 mengontrol dampak kompresi; frozen v6 menjadi referensi guard. Seluruh benchmark sudah teramati, bukan test baru atau validasi Indonesia. Model internal tidak diganti.

Gunakan environment Python 3.12 proyek. Run All pertama melatih; berikutnya memverifikasi dan memuat artifact beku tanpa tuning ulang. Dataset/model lokal tidak disertakan Git.

In [1]:
import gc
import json
import time
import warnings
from datetime import datetime, timezone
import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier, Pool
from scipy.special import logit
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from guardian_ml.paths import ROOT
from guardian_ml.metadata import checksum
from guardian_ml.family_features import (
    fit_shared_encoder,
    shared_matrix,
    source_weights,
)
from guardian_ml.family_scores import raw_scores, matrix_raw, calibrated, probabilities
from guardian_ml.family_audit import (
    write_json,
    value_hash,
    code_hash,
    audit_development,
    assert_external,
)
from guardian_ml.structural_evaluation import (
    safe_metrics,
    source_point,
    paired_domain_interval,
    selection_diagnostics,
)
from guardian_ml.v8_selection import guarded_diagnostics, choose_candidate

SEED = 20261101
RUN = ROOT / "ml/runs/notebook-model-comparison-v9"
REPORTS, DATA, ARTIFACTS = RUN / "reports", RUN / "data", RUN / "artifacts"
for folder in [REPORTS, DATA, ARTIFACTS]:
    folder.mkdir(parents=True, exist_ok=True)
NOTEBOOK = ROOT / "guardian_lab_url_qr_comparison_v9.ipynb"
FROZEN = REPORTS / "frozen-v9.json"
REUSE = FROZEN.exists()
V8 = ROOT / "ml/runs/notebook-tfidf-logistic-v8"
V6 = ROOT / "ml/runs/notebook-tfidf-logistic-v6"
DEV_PATH = V8 / "data/development-v8.parquet"
CONFIGS = [
    {"name": "lr_shared_c05", "family": "shared_lr", "C": 0.5},
    {"name": "lr_shared_c2", "family": "shared_lr", "C": 2.0},
    {"name": "catboost_shared_depth6", "family": "shared_catboost", "depth": 6},
    {"name": "catboost_shared_depth8", "family": "shared_catboost", "depth": 8},
]


def now():
    return datetime.now(timezone.utc).isoformat()


source_names = [
    "family_features",
    "family_scores",
    "family_audit",
    "structural_features",
    "structural_evaluation",
    "rich_features",
    "features",
    "preprocessing",
    "evaluation",
    "v8_selection",
]
source_hashes = {
    "ml/src/guardian_ml/" + name + ".py": checksum(
        ROOT / "ml/src/guardian_ml" / (name + ".py")
    )
    for name in source_names
}
protected = {}
for v, notebook in [
    (3, "guardian_lab_url_qr.ipynb"),
    (6, "guardian_lab_url_qr_data_v6.ipynb"),
    (7, "guardian_lab_url_qr_structural_v7.ipynb"),
    (8, "guardian_lab_url_qr_expansion_v8.ipynb"),
]:
    folder = ROOT / f"ml/runs/notebook-tfidf-logistic-v{v}"
    original = json.loads((folder / f"reports/frozen-v{v}.json").read_text())
    path = folder / f"artifacts/url-core-logistic-v{v}.joblib"
    assert checksum(path) == original["bundle_sha256"]
    for p in [path, folder / f"reports/frozen-v{v}.json", ROOT / notebook]:
        protected[str(p.relative_to(ROOT))] = checksum(p)
protected["ml/policies/shadow-runtime-manifest.json"] = checksum(
    ROOT / "ml/policies/shadow-runtime-manifest.json"
)
v8 = joblib.load(V8 / "artifacts/url-core-logistic-v8.joblib")
v8["family"] = "full_v8"
v6 = joblib.load(V6 / "artifacts/url-core-logistic-v6.joblib")
v6["family"] = "full_v6"
print("Offline comparison: no target requests, no new labels, shadow v6 preserved.")

Offline comparison: no target requests, no new labels, shadow v6 preserved.


## Audit, partisi dan protokol beku
Hash data parent, implementasi dan konfigurasi diikat sebelum fitting. Identitas domain/tenant, digest dan canonical core diverifikasi; semua peran development terpisah. Tidak menampilkan baris URL.

In [2]:
development = pd.read_parquet(DEV_PATH)
audit = audit_development(development)
roles = {
    role: part.reset_index(drop=True) for role, part in development.groupby("partition")
}
fit, tune, calibration, threshold_data = [
    roles[k] for k in ["fit", "tune", "calibration", "threshold"]
]
spec = {
    "configs": CONFIGS,
    "development_sha256": checksum(DEV_PATH),
    "code_sha256": code_hash(NOTEBOOK),
    "source_hashes": source_hashes,
    "protocol_md_sha256": checksum(
        ROOT / "ai-agent-docs/model-family-comparison-v9-plan.md"
    ),
    "protected": protected,
    "seed": SEED,
    "shared": {
        "svd_components": 64,
        "svd_iter": 5,
        "source_cap": 12000,
        "sample_cap": 60000,
        "structure_features": 38,
        "vocabulary": "frozen_v8_fit_only",
    },
    "source_weight_power": 1.0,
    "catboost": {
        "iterations": 600,
        "learning_rate": 0.06,
        "l2_leaf_reg": 5,
        "early_stopping_rounds": 60,
        "thread_count": 4,
    },
    "guard_margin": 0.01,
    "selection": "guard_then_min_source_recall_macro_AP",
    "benchmarks_status": "all_previously_observed",
    "versions": {
        key: __import__("importlib.metadata", fromlist=["version"]).version(key)
        for key in ["numpy", "pandas", "scikit-learn", "catboost", "scipy"]
    },
}
benchmark_paths = {
    "observed_phish360": ROOT
    / "ml/runs/notebook-tfidf-logistic-v7/data/phish360-new.parquet",
    "observed_tr_op": V8 / "data/tr-op-test-v8.parquet",
    "observed_vendor": V6 / "data/deepurlbench-holdout-v6.parquet",
    "observed_human_phishvn": ROOT
    / "ml/runs/notebook-tfidf-logistic-v5/data/phishvn-human-new.parquet",
    "observed_phishtank_positive": V6 / "data/phishtank-2026-v6.parquet",
    "observed_phresh_parent": ROOT
    / "ml/runs/notebook-tfidf-logistic-v3/data/development-v3.parquet",
    "observed_urlphish": ROOT
    / "ml/runs/notebook-tfidf-logistic-v3/data/urlphish-fresh.parquet",
}
spec["benchmark_assets"] = {
    name: {"path": str(path.relative_to(ROOT)), "sha256": checksum(path)}
    for name, path in benchmark_paths.items()
}
spec_sha = value_hash(spec)
if REUSE:
    frozen = json.loads(FROZEN.read_text())
    assert frozen["spec_sha256"] == spec_sha
    validation = json.loads((REPORTS / "validation-results.json").read_text())
else:
    assert not (REPORTS / "evaluation-v9.json").exists()
    protocol_path = REPORTS / "notebook-protocol.json"
    if protocol_path.exists():
        assert json.loads(protocol_path.read_text())["spec_sha256"] == spec_sha
    else:
        write_json(
            protocol_path,
            {
                "declared_at_utc": now(),
                "spec_sha256": spec_sha,
                "specification": spec,
                "data_audit": audit,
                "benchmark_predictions_in_this_run": False,
            },
        )
display(
    pd.DataFrame(
        [
            {
                "partition": k,
                "rows": len(v),
                "phishing": int(v.label.sum()),
                "benign": int((v.label == 0).sum()),
            }
            for k, v in roles.items()
        ]
    )
)
print("Data/spec/code verified before training.")

,partition,rows,phishing,benign
0,calibration,40635,14380,26255
1,fit,535504,201161,334343
2,threshold,47219,15086,32133
3,tune,84545,30884,53661


Data/spec/code verified before training.


## Representasi bersama fit-only
TruncatedSVD dilatih hanya dari sampel fitting, menggunakan vocabulary frozen v8 yang dilatih pada fitting yang sama. Transformasi batch menghasilkan 102 fitur tanpa skema/query/source ID. Kompresi dapat kehilangan pola langka; kontrol LR lengkap tetap dilaporkan.

In [3]:
if not REUSE:
    started = time.perf_counter()
    with threadpool_limits(limits=4):
        shared = fit_shared_encoder(
            v8["encoder"], fit.core, fit.training_source, partition="fit"
        )
        joblib.dump(shared, ARTIFACTS / "shared-encoder.joblib", compress=3)
        feature_checks = {}
        for role, frame in roles.items():
            print("Transform development", role, len(frame), flush=True)
            matrix = shared_matrix(shared, frame.core)
            path = DATA / (role + "-shared.npy")
            np.save(path, matrix)
            feature_checks[role] = {
                "rows": len(matrix),
                "columns": matrix.shape[1],
                "sha256": checksum(path),
            }
            del matrix
            gc.collect()
    feature_report = {
        "seconds": time.perf_counter() - started,
        "svd_sample_rows": shared["svd_sample_rows"],
        "explained_variance_ratio_sum": float(
            shared["svd"].explained_variance_ratio_.sum()
        ),
        "encoder_sha256": checksum(ARTIFACTS / "shared-encoder.joblib"),
        "matrices": feature_checks,
    }
    write_json(REPORTS / "feature-audit.json", feature_report)
else:
    feature_report = json.loads((REPORTS / "feature-audit.json").read_text())
print(
    "Feature preparation:",
    feature_report["seconds"],
    "seconds; SVD fit rows:",
    feature_report["svd_sample_rows"],
)

Transform development calibration 40635


Transform development fit 535504


Transform development threshold 47219


Transform development tune 84545


Feature preparation: 147.0884011000162 seconds; SVD fit rows: 52636


## Pelatihan dua konfigurasi per keluarga
Semua classifier baru dilatih pada 535.504 fitting dengan bobot sumber sama. CatBoost early stopping hanya menggunakan tune. Pilih satu finalist per keluarga berdasarkan guard dan minimum recall sumber, bukan accuracy benchmark.

In [4]:
if not REUSE:
    train_x = np.load(DATA / "fit-shared.npy", mmap_mode="r")
    tune_x = np.load(DATA / "tune-shared.npy", mmap_mode="r")
    weights = source_weights(fit.training_source)
    tune_weights = source_weights(tune.training_source)
    with threadpool_limits(limits=4):
        reference = selection_diagnostics(tune, raw_scores(v6, tune.core))
        full_v8_diagnostic = guarded_diagnostics(
            tune, raw_scores(v8, tune.core), reference
        )
    scaler = StandardScaler().fit(train_x)
    validation = []
    for config in CONFIGS:
        print("Fit", config["name"], flush=True)
        started = time.perf_counter()
        if config["family"] == "shared_lr":
            classifier = LogisticRegression(
                C=config["C"],
                solver="liblinear",
                max_iter=1500,
                tol=0.0001,
                random_state=SEED,
            )
            with (
                warnings.catch_warnings(record=True) as captured,
                threadpool_limits(limits=4),
            ):
                warnings.simplefilter("always", ConvergenceWarning)
                classifier.fit(
                    scaler.transform(train_x), fit.label, sample_weight=weights
                )
            converged = not any(
                issubclass(w.category, ConvergenceWarning) for w in captured
            )
            candidate = {
                "family": config["family"],
                "classifier": classifier,
                "scaler": scaler,
                "shared_encoder": shared,
            }
            iterations = int(classifier.n_iter_[0])
        else:
            classifier = CatBoostClassifier(
                iterations=600,
                depth=config["depth"],
                learning_rate=0.06,
                l2_leaf_reg=5,
                loss_function="Logloss",
                eval_metric="Logloss",
                random_seed=SEED,
                thread_count=4,
                allow_writing_files=False,
                verbose=False,
                used_ram_limit="1536mb",
            )
            classifier.fit(
                Pool(train_x, fit.label, weight=weights),
                eval_set=Pool(tune_x, tune.label, weight=tune_weights),
                early_stopping_rounds=60,
                use_best_model=True,
                verbose=False,
            )
            candidate = {
                "family": config["family"],
                "classifier": classifier,
                "shared_encoder": shared,
            }
            iterations = int(classifier.tree_count_)
            converged = True
        seconds = time.perf_counter() - started
        with threadpool_limits(limits=4):
            raw = matrix_raw(candidate, tune_x)
        diagnostic = guarded_diagnostics(tune, raw, reference)
        diagnostic.update(
            {
                "name": config["name"],
                "family": config["family"],
                "config": config,
                "fit_seconds": seconds,
                "iterations_or_trees": iterations,
                "completed_without_convergence_warning": converged,
            }
        )
        if not converged:
            diagnostic["source_guard_passed"] = False
        validation.append(diagnostic)
        joblib.dump(candidate, ARTIFACTS / (config["name"] + ".joblib"), compress=3)
        write_json(REPORTS / "validation-results.json", validation)
        print(
            "Fit complete",
            config["name"],
            round(seconds, 2),
            "seconds; macro recall",
            round(diagnostic["macro_source_recall"], 4),
            "min recall",
            round(diagnostic["min_source_recall"], 4),
            "guard",
            diagnostic["source_guard_passed"],
            flush=True,
        )
        del candidate, classifier
        gc.collect()
    finalists = {}
    for family in ["shared_lr", "shared_catboost"]:
        selected, decision = choose_candidate(
            [r for r in validation if r["family"] == family]
        )
        finalists[family] = {"selected": selected["name"], "decision": decision}
    overall, overall_decision = choose_candidate(validation)
    selection = {
        "finalists": finalists,
        "overall_development": overall_decision,
        "full_v8_diagnostic": full_v8_diagnostic,
        "v6_reference": reference,
        "runtime_retained": "frozen_v6",
        "production_promotion": False,
    }
    write_json(REPORTS / "selection-before-benchmark.json", selection)
    del train_x, tune_x, scaler
    gc.collect()
else:
    selection = json.loads((REPORTS / "selection-before-benchmark.json").read_text())
display(
    pd.DataFrame(
        [
            {
                k: r[k]
                for k in [
                    "name",
                    "fit_seconds",
                    "macro_source_recall",
                    "min_source_recall",
                    "source_guard_passed",
                ]
            }
            for r in validation
        ]
    )
)

Fit lr_shared_c05


Fit complete lr_shared_c05 32.4 seconds; macro recall 0.222 min recall 0.0322 guard False


Fit lr_shared_c2


Fit complete lr_shared_c2 34.64 seconds; macro recall 0.2215 min recall 0.0322 guard False


Fit catboost_shared_depth6


Fit complete catboost_shared_depth6 84.11 seconds; macro recall 0.3259 min recall 0.185 guard False


Fit catboost_shared_depth8


Fit complete catboost_shared_depth8 117.2 seconds; macro recall 0.3578 min recall 0.2547 guard False


,name,fit_seconds,macro_source_recall,min_source_recall,source_guard_passed
0,lr_shared_c05,32.399004,0.222009,0.032172,False
1,lr_shared_c2,34.635142,0.221496,0.032172,False
2,catboost_shared_depth6,84.113111,0.325930,0.184987,False
3,catboost_shared_depth8,117.202318,0.357764,0.254692,False


## Kalibrasi dan threshold setara sebelum benchmark
Setiap finalist dan kontrol memakai calibration dan threshold partition yang sama. Ambang numeriknya berbeda, tetapi aturan batas FPR per sumber sama. Kebijakan asli baseline tetap tersimpan terpisah. Bekukan seluruh model/policy sebelum evaluasi.

In [5]:
if not REUSE:
    bundles = {
        name: joblib.load(ARTIFACTS / (info["selected"] + ".joblib"))
        for name, info in finalists.items()
    }
    bundles.update({"full_v8": v8, "full_v6": v6})
    frozen_models = {}
    for name, bundle in bundles.items():
        print("Calibrate and freeze", name, flush=True)
        with threadpool_limits(limits=4):
            if name.startswith("shared_"):
                cal_raw = matrix_raw(
                    bundle, np.load(DATA / "calibration-shared.npy", mmap_mode="r")
                )
                threshold_raw = matrix_raw(
                    bundle, np.load(DATA / "threshold-shared.npy", mmap_mode="r")
                )
            else:
                cal_raw = raw_scores(bundle, calibration.core)
                threshold_raw = raw_scores(bundle, threshold_data.core)
            calibrator = LogisticRegression(C=1000, max_iter=500, random_state=SEED)
            calibrator.fit(
                logit(np.clip(cal_raw, 1e-7, 1 - 1e-7)).reshape(-1, 1),
                calibration.label,
            )
            assert calibrator.coef_[0, 0] > 0
            bundle["calibrator"] = calibrator
            p = calibrated(bundle, threshold_raw)
        policy = {
            level: source_point(threshold_data, p, budget, confidence=True)
            for level, budget in [("warning", 0.01), ("high", 0.001)]
        }
        thresholds = {level: point["threshold"] for level, point in policy.items()}
        bundle["comparison_metadata"] = {
            "name": name,
            "spec_sha256": spec_sha,
            "code_sha256": spec["code_sha256"],
            "thresholds": thresholds,
            "threshold_policy": policy,
            "fit_rows": len(fit) if name != "full_v6" else 532793,
            "classifier_refitted": name.startswith("shared_"),
            "calibration_refitted_on_v9_role": True,
            "production_ready": False,
            "benchmarks_previously_observed": True,
        }
        path = ARTIFACTS / (name + "-frozen.joblib")
        joblib.dump(bundle, path, compress=3)
        restored = joblib.load(path)
        probes = [
            "http://example.org/login",
            "https://example.org/login",
            "https://alice.github.io/account",
        ]
        with threadpool_limits(limits=4):
            a = probabilities(bundle, probes)
            b = probabilities(restored, probes)
        assert np.array_equal(a, b) and a[0] == a[1] and np.isfinite(a).all()
        frozen_models[name] = {
            "path": str(path.relative_to(ROOT)),
            "sha256": checksum(path),
            "metadata": bundle["comparison_metadata"],
            "save_reload_and_scheme_parity": True,
        }
    frozen = {
        "frozen_at_utc": now(),
        "spec_sha256": spec_sha,
        "specification": spec,
        "models": frozen_models,
        "selection": selection,
        "benchmark_predictions_in_this_run": False,
        "protected": protected,
    }
    write_json(FROZEN, frozen)
else:
    bundles = {}
    for name, item in frozen["models"].items():
        assert checksum(ROOT / item["path"]) == item["sha256"]
        bundles[name] = joblib.load(ROOT / item["path"])
assert code_hash(NOTEBOOK) == frozen["specification"]["code_sha256"]
assert all(checksum(ROOT / p) == digest for p, digest in frozen["protected"].items())
print("Four comparison policies frozen. Runtime unchanged.")

Calibrate and freeze shared_lr


Calibrate and freeze shared_catboost


Calibrate and freeze full_v8


Calibrate and freeze full_v6


Four comparison policies frozen. Runtime unchanged.


## Evaluasi benchmark yang sudah teramati
Semua model dinilai pada baris yang sama. Benchmark tetap di luar fitting/selection/calibration/threshold. Feed positif hanya memberi recall; metrik accuracy/FPR/precision trafik tidak tersedia. Hasil test tidak memilih model pengganti.

In [6]:
historical = pd.read_parquet(
    ROOT / "ml/runs/notebook-tfidf-logistic-v3/data/development-v3.parquet"
)
benchmarks = {
    "observed_phish360": pd.read_parquet(
        ROOT / "ml/runs/notebook-tfidf-logistic-v7/data/phish360-new.parquet"
    ),
    "observed_tr_op": pd.read_parquet(V8 / "data/tr-op-test-v8.parquet"),
    "observed_vendor": pd.read_parquet(V6 / "data/deepurlbench-holdout-v6.parquet"),
    "observed_human_phishvn": pd.read_parquet(
        ROOT / "ml/runs/notebook-tfidf-logistic-v5/data/phishvn-human-new.parquet"
    ),
    "observed_phishtank_positive": pd.read_parquet(
        V6 / "data/phishtank-2026-v6.parquet"
    ),
    "observed_phresh": historical[
        historical.partition.eq("test") & historical.novel_domain.eq(True)
    ].reset_index(drop=True),
    "observed_urlphish": pd.read_parquet(
        ROOT / "ml/runs/notebook-tfidf-logistic-v3/data/urlphish-fresh.parquet"
    ),
}
del historical
gc.collect()
evaluation = {}
paired = {}
rows = []
for dataset, frame in benchmarks.items():
    frame = frame.reset_index(drop=True)
    assert_external(frame, development)
    print("Evaluate observed benchmark", dataset, len(frame), flush=True)
    benchmark_scores = {}
    model_metrics = {}
    with threadpool_limits(limits=4):
        shared_x = shared_matrix(bundles["shared_lr"]["shared_encoder"], frame.core)
        for name, bundle in bundles.items():
            started = time.perf_counter()
            raw = (
                matrix_raw(bundle, shared_x)
                if name.startswith("shared_")
                else raw_scores(bundle, frame.core)
            )
            p = calibrated(bundle, raw)
            elapsed = time.perf_counter() - started
            benchmark_scores[name] = p
            policy = bundle["comparison_metadata"]["thresholds"]
            levels = {
                level: safe_metrics(frame.label, p, t) for level, t in policy.items()
            }
            for item in levels.values():
                item["balanced_accuracy"] = (
                    (item["recall"] + 1 - item["fpr"]) / 2
                    if item["fpr"] is not None and item["recall"] is not None
                    else None
                )
            model_metrics[name] = {
                "levels": levels,
                "scoring_seconds_excluding_shared_transform": elapsed,
            }
            item = levels["warning"]
            rows.append(
                {
                    "dataset": dataset,
                    "model": name,
                    "rows": len(frame),
                    **{
                        key: None if item[key] is None else 100 * item[key]
                        for key in [
                            "accuracy",
                            "recall",
                            "fpr",
                            "precision",
                            "balanced_accuracy",
                        ]
                    },
                }
            )
    del shared_x
    gc.collect()
    paired[dataset] = {
        reference: paired_domain_interval(
            frame,
            benchmark_scores[reference],
            benchmark_scores["shared_catboost"],
            bundles[reference]["comparison_metadata"]["thresholds"]["warning"],
            bundles["shared_catboost"]["comparison_metadata"]["thresholds"]["warning"],
            repeats=1000,
        )
        for reference in ["full_v8", "shared_lr"]
    }
    evaluation[dataset] = {
        "rows": len(frame),
        "domains": int(frame.group.nunique()),
        "phishing": int(frame.label.sum()),
        "benign": int((frame.label == 0).sum()),
        "previously_observed": True,
        "used_for_v9_training_selection_calibration_threshold": False,
        "models": model_metrics,
    }
results_table = pd.DataFrame(rows)
report = {
    "completed_at_utc": now(),
    "protocol": "ai-agent-docs/model-family-comparison-v9-plan.md",
    "notebook": NOTEBOOK.name,
    "frozen_sha256": checksum(FROZEN),
    "spec_sha256": spec_sha,
    "data_audit": audit,
    "feature_audit": feature_report,
    "validation_results": validation,
    "selection": selection,
    "benchmarks": evaluation,
    "paired_domain_intervals": paired,
    "models": frozen["models"],
    "production_ready": False,
    "new_independent_test": False,
    "indonesia_gold_available": False,
}
write_json(REPORTS / "evaluation-v9.json", report)
write_json(ROOT / "ml/reports/model-comparison-v9.json", report)
results_table.to_csv(REPORTS / "benchmark-table.csv", index=False)
display(results_table.round(3))

Evaluate observed benchmark observed_phish360 7516


Evaluate observed benchmark observed_tr_op 1097


Evaluate observed benchmark observed_vendor 20000


Evaluate observed benchmark observed_human_phishvn 33


Evaluate observed benchmark observed_phishtank_positive 9269


Evaluate observed benchmark observed_phresh 80052


Evaluate observed benchmark observed_urlphish 85824


,dataset,model,rows,accuracy,recall,fpr,precision,balanced_accuracy
0,observed_phish360,shared_lr,7516,64.715,17.432,0.209,98.413,58.612
1,observed_phish360,shared_catboost,7516,73.922,38.988,0.162,99.442,69.413
2,observed_phish360,full_v8,7516,78.060,48.891,0.301,99.176,74.295
3,observed_phish360,full_v6,7516,72.938,36.614,0.116,99.575,68.249
4,observed_tr_op,shared_lr,1097,52.963,27.770,0.260,99.497,63.755
5,observed_tr_op,shared_catboost,1097,65.725,47.265,0.000,100.000,73.633
6,observed_tr_op,full_v8,1097,73.291,58.906,0.000,100.000,79.453
7,observed_tr_op,full_v6,1097,60.438,39.130,0.000,100.000,69.565
8,observed_vendor,shared_lr,20000,90.780,21.845,0.652,80.634,60.597
9,observed_vendor,shared_catboost,20000,91.875,31.479,0.618,86.352,65.430


## Visualisasi dan verifikasi akhir
Recall dan false alarm harus dibaca bersama. Perbandingan fitur bersama tidak mencakup semua representasi CatBoost; SVD dapat menghilangkan sinyal. Kalibrasi benchmark tidak membuktikan probabilitas keamanan pada trafik nyata.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
chosen = results_table[
    results_table.dataset.isin(
        ["observed_phish360", "observed_tr_op", "observed_vendor", "observed_phresh"]
    )
]
for axis, metric in zip(axes, ["recall", "fpr"]):
    chosen.pivot(index="dataset", columns="model", values=metric).plot.bar(ax=axis)
    axis.set_ylabel(metric + " (%)")
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(REPORTS / "comparison-v9.png", dpi=160)
display(fig)
plt.close(fig)
checks = {
    "notebook_code_matches_frozen": code_hash(NOTEBOOK)
    == frozen["specification"]["code_sha256"],
    "protected_artifacts_unchanged": all(
        checksum(ROOT / p) == digest for p, digest in frozen["protected"].items()
    ),
    "all_frozen_comparison_models_unchanged": all(
        checksum(ROOT / item["path"]) == item["sha256"]
        for item in frozen["models"].values()
    ),
    "no_production_promotion": not selection["production_promotion"],
    "no_independent_test_claim": not report["new_independent_test"],
}
assert all(checks.values())
write_json(REPORTS / "completion-checks.json", checks)
print("Completed:", checks)
print(
    "Development decision:", selection["overall_development"]["development_selection"]
)
print("Runtime remains frozen v6. Indonesia validation remains unavailable.")

<Figure size 1300x500 with 2 Axes>

Completed: {'notebook_code_matches_frozen': True, 'protected_artifacts_unchanged': True, 'all_frozen_comparison_models_unchanged': True, 'no_production_promotion': True, 'no_independent_test_claim': True}
Development decision: retain_frozen_v6
Runtime remains frozen v6. Indonesia validation remains unavailable.
